Compare preidction accuracy in first vs last 20 mins

In [ ]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

experiments_objects = [JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr,
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

In [ ]:
from behave_analysis.process.session import get_experiment 
from JR_test_scripts.tracking2features import tracking_to_features, extract_pa_across_sesh

import os
import dill as pickle
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
# fixed vars
ceph_path = r"Z:\Jasmine_Laurence\Experimental_Data"
winstor_path = r"Y:\Laurence"
save_path = r"Z:\Jasmine_Laurence\LDA_overview"
conditions = ['shelter_only','barrier_pre_flip','barrier_post_flip']
all_angles = ['hdir','hsa','h_preflipbar_a','h_postflipbar_a','h_rightbar_a','h_leftbar_a']
all_features = ['shelter','preflip_barrier','postflip_barrier','left_barrier', 'right_barrier', 'arena_bottom', 'arena_top']

In [ ]:
# pulling out prediction accuracy for all sessions for a certain set of settings
settings = "all_angles_fr"  # has only be computed in the non subsampled one
time_cond = ["first_half", "second_half"]
conditions_h = conditions  # ['barrier_pre_flip','barrier_post_flip']

sessy, all_names, avg_pa, pa = extract_pa_across_sesh(
    all_angles, experiments_objects, conditions_h, all_features, settings, bar_angles=False, time_cond=time_cond
)

In [ ]:
''' Plot the difference in pa for two goals'''
combos = [['h_preflipbar_a','h_postflipbar_a'],['h_rightbar_a','h_leftbar_a']]

plt.rcParams['figure.figsize'] = [15, 5*len(conditions_h)]
fig, axs = plt.subplots(len(conditions_h),len(combos))

for idx,c in enumerate(combos):
    for idx_c,cd in enumerate(conditions_h):
        diff = pa[c[0]][:,idx_c,:] - pa[c[1]][:,idx_c,:]
        axs[idx_c,idx].bar(time_cond,np.nanmean(diff,axis=0),color = 'k',alpha = .3)
        axs[idx_c,idx].set_prop_cycle(plt.cycler('color', plt.cm.hsv(np.linspace(0, 1, len(pa[c[0]])))))
        for p in diff:
            axs[idx_c,idx].plot(time_cond,p)
        axs[idx_c,idx].set_title(c[0] + ' - ' + c[1]  +  
                        '\n in ' + cd,
                        fontsize = 10)
        axs[idx_c,idx].plot([-.5,1.5],[0,0],'--k')
        axs[idx_c,idx].set_xticks(np.arange(len(time_cond)))
        axs[idx_c,idx].set_xticklabels(time_cond,rotation = 45, ha="right")
        axs[idx_c,idx].set_ylim([-.15,.15])
        axs[idx_c,0].set_ylabel('diff in avg. prediction accuracy')
axs[idx_c,idx].legend(all_names,loc='center left', bbox_to_anchor=(1.2, 0.5),
                columnspacing=1.0, labelspacing=0.2,
                handletextpad=0.5, handlelength=1.5)
plt.tight_layout()
plt.savefig(save_path+'/'+str('pa_diff_allsesh_'+settings+'_time_cond_allcond.png'))

In [ ]:
plt.rcParams['figure.figsize'] = [7,5]
fig, axs = plt.subplots(1,1)
angle = ['h_preflipbar_a','h_postflipbar_a']
aligned = []
name = []
for c_idx, c in enumerate(conditions):
    for t_idx, t in enumerate(time_cond):
        if len(aligned) == 0:
            aligned = pa[angle[0]][:,c_idx,t_idx] - pa[angle[1]][:,c_idx,t_idx]
        else:
            aligned = np.vstack((aligned, pa[angle[0]][:,c_idx,t_idx] - pa[angle[1]][:,c_idx,t_idx]))
        name.append(str(c + '\n' + t))

axs.set_prop_cycle(plt.cycler('color', plt.cm.hsv(np.linspace(0, 1, len(pa[angle[0]])))))
axs.plot(aligned)
axs.plot(aligned[:,15],'k') # what is up with this JAL6 session?!
axs.plot([-.5,np.shape(aligned)[0]-.5],[0,0],'--k')
axs.legend(all_names,
                loc='center left', 
                bbox_to_anchor=(1.1, .5),
                columnspacing=1.0, labelspacing=0.2,
                handletextpad=0.5, handlelength=1.5)
axs.set_xticks(np.arange(np.shape(aligned)[0]))
axs.set_xticklabels(name,rotation = 45)
axs.set_ylabel('preflip - postflip \ndiff in prediction accuracy')
plt.tight_layout()
plt.savefig(save_path+'/'+str('pre_post_diff_pa_time_in_sesh.png'))

In [ ]:
'''Plot diff in prediction accuracy in first vs second half averaged by mouse'''
plt.rcParams['figure.figsize'] = [7,5]
fig, axs = plt.subplots(1,1)
angle = ['h_preflipbar_a','h_postflipbar_a']
aligned = []
name = []
for c_idx, c in enumerate(conditions):
    for t_idx, t in enumerate(time_cond):
        if len(aligned) == 0:
            aligned = pa[angle[0]][:,c_idx,t_idx] - pa[angle[1]][:,c_idx,t_idx]
        else:
            aligned = np.vstack((aligned, pa[angle[0]][:,c_idx,t_idx] - pa[angle[1]][:,c_idx,t_idx]))
        name.append(str(c + '\n' + t))

aligned = np.delete(aligned,15,1)
not_all_names = all_names[:15] + all_names[16:]
mouse_ID = [x[:6] for x in not_all_names]
unique_mouse = [x for i, x in enumerate(mouse_ID) if x not in mouse_ID[:i]]
mean_aligned = np.zeros((len(unique_mouse),np.shape(aligned)[0]))
for m_idx, m in enumerate(unique_mouse):
    this_mouse = [i for i,x in enumerate(mouse_ID) if x == m]
    mean_aligned[m_idx,:] = np.mean(aligned[:,this_mouse], axis=1)

axs.plot(mean_aligned.T)
axs.plot([-.5,np.shape(aligned)[0]-.5],[0,0],'--k')
axs.legend(unique_mouse,
                loc='center left', 
                bbox_to_anchor=(1.1, .5),
                columnspacing=1.0, labelspacing=0.2,
                handletextpad=0.5, handlelength=1.5)
axs.set_xticks(np.arange(np.shape(aligned)[0]))
axs.set_xticklabels(name,rotation = 45)
axs.set_ylabel('preflip - postflip \ndiff in prediction accuracy')
plt.tight_layout()
plt.savefig(save_path+'/'+str('pre_post_diff_pa_time_in_sesh_mouse_avg.png'))

In [ ]:
''' Plot the difference in pa (averaged over an area) for two goals'''
combos = [['preflip_barrier','postflip_barrier'],['left_barrier', 'right_barrier'], ['arena_bottom', 'arena_top']]

plt.rcParams['figure.figsize'] = [15, 5*len(conditions_h)]
fig, axs = plt.subplots(len(conditions_h),len(combos))

for idx,c in enumerate(combos):
    for idx_c,cd in enumerate(conditions_h):
        diff = avg_pa[c[0]][:,idx_c,:] - avg_pa[c[1]][:,idx_c,:]
        axs[idx_c,idx].bar(time_cond,np.nanmean(diff,axis=0),color = 'k',alpha = .3)
        axs[idx_c,idx].set_prop_cycle(plt.cycler('color', plt.cm.hsv(np.linspace(0, 1, len(avg_pa[c[0]])))))
        for p in diff:
            axs[idx_c,idx].plot(time_cond,p)
        axs[idx_c,idx].set_title(c[0] + ' - ' + c[1]  +  
                        '\n in ' + cd,
                        fontsize = 10)
        axs[idx_c,idx].plot([-.5,1.5],[0,0],'--k')
        axs[idx_c,idx].set_xticks(np.arange(len(time_cond)))
        axs[idx_c,idx].set_xticklabels(time_cond,rotation = 45, ha="right")
        axs[idx_c,idx].set_ylim([-.15,.15])
        axs[idx_c,0].set_ylabel('diff in avg. prediction accuracy')
axs[idx_c,idx].legend(all_names,loc='center left', bbox_to_anchor=(1.2, 0.5),
                columnspacing=1.0, labelspacing=0.2,
                handletextpad=0.5, handlelength=1.5)
plt.tight_layout()
plt.savefig(save_path+'/'+str('AVGpa_diff_allsesh_'+settings+'_time_cond_allcond.png'))